# Øvelser: Fra seks scrapere til ét korpus

**Social Data Science 1 — lektion 6**

I får ikke et færdigt datasæt i dag. I får seks rå filer, lavet af seks forskellige
scrapere, og første opgave er at gøre dem til én tabel.

Det er ikke oprydning før det rigtige arbejde. Det **er** det rigtige arbejde — og
hvert valg undervejs er en beslutning om, hvad korpusset kan bruges til.

**Materialet** er indlæg fra seks danske højrefløjsmedier og -foreninger, indsamlet
i 2022-2023. Vi analyserer det som forskningsmateriale.

**Filer:**

```
DKSamling_20230627.json          Frihedens_stemme_20221004.json
SIAD_20230202.json               nordfront_20230202.json
trykkefrihed_20220824.json       uriaspost_20230306.csv
```


---

## Opgave 1: Kig på én fil ad gangen

**Trin 1 — åbn en JSON-fil.**

JSON er ikke en tabel. Det er en indlejret struktur, som kan indeholde lister og
dictionaries i hinanden. `json.load()` giver jer den som Python-objekter.

Kør cellen. Hvad er `d` for en datastruktur, og hvor mange elementer er der i den?


In [ ]:
import json
import pandas as pd

d = json.load(open("SIAD_20230202.json", encoding="utf-8"))

print(type(d))
print(len(d))
print(d[0].keys())


**Trin 2 — kig på ét dokument.**

`d[0]` er den første post. Print de enkelte felter, så I kan se, hvad der er i dem.

*Hint:* `d[0]["post_date"]` — først liste-indeks, så dictionary-nøgle. Samme mønster
som i lektion 3.


In [ ]:
# Din kode her



**Trin 3 — de andre fem filer.**

Åbn hver af de fire andre JSON-filer og print deres nøgler. Skriv ned, hvilke felter
der svarer til hinanden på tværs af kilderne.

Til sidst CSV-filen:

```python
u = pd.read_csv("uriaspost_20230306.csv")
```

Den fejler. Læs fejlbeskeden, og find ud af hvorfor.

*Hint:* åbn filen i en teksteditor og kig på den første linje.


In [ ]:
# Din kode her



### 1.4 Til diskussion

1. Hvilke felter findes i alle seks kilder? Hvilke findes kun i nogle?
2. SIAD har ingen overskrift. Betyder det, at indlæggene ikke havde en?
3. Hvad er forskellen på et manglende felt og et tomt felt?


---

## Opgave 2: Byg korpusset

**Trin 1 — én kilde først.**

Byg en liste af dictionaries med de samme fire nøgler for hver post, og lav den om
til en DataFrame. Præcis mønstret fra lektion 3.

Udfyld de to pladser.


In [ ]:
raekker = []

for r in d:
    raekker.append({
        "kilde": "SIAD",
        "dato_raa": ______,
        "tekst": ______
    })

siad = pd.DataFrame(raekker)
print(siad.shape)
print(siad.head(3))


**Trin 2 — gør det til en funktion.**

I skal gøre det samme seks gange med forskellige feltnavne. Det er en funktion.

```python
def indlaes(fil, kilde, dato_felt, tekst_felt):
    # åbn filen
    # byg listen
    # returnér en DataFrame
```

Bemærk, at to af kilderne **ikke har** et datofelt. Hvordan håndterer funktionen det?


In [ ]:
# Din kode her



**Trin 3 — saml dem.**

Brug `pd.concat()` til at lægge de seks tabeller sammen. Husk `ignore_index=True`.

Hvor mange rækker får I i alt? Og hvordan fordeler de sig på kilder?

*Hint:* `korpus["kilde"].value_counts()`


In [ ]:
# Din kode her



### 2.4 Til diskussion

1. Gæt, før I kører `value_counts()`: hvilken kilde fylder mest?
2. Hvor mange dokumenter har helt tom tekst? Hvad gør I med dem?
3. I har nu truffet mindst fem beslutninger, som ikke står nogen steder. Skriv dem ned.


---

## Opgave 3: Datoerne

Fire kilder har datoer, i fire forskellige formater:

```
Dansk Samling       31. december 2019
Frihedens Stemme    23. januar 2019
SIAD                januar 1, 2019
Nordfront           2 februar, 2023
```

**Trin 1 — prøv den nemme vej.**

Kør cellen. Hvor mange lykkes det for?


In [ ]:
korpus["dato"] = pd.to_datetime(korpus["dato_raa"], errors="coerce")

print("Fik en dato:", korpus["dato"].notna().sum(), "af", len(korpus))
print(korpus.groupby("kilde")["dato"].count())


**Trin 2 — månedsnavnene.**

`pd.to_datetime()` kender ikke danske månedsnavne. Byg en dictionary, der oversætter
dem til tal, og brug den til at rette teksten, før I konverterer.

```python
maaneder = {"januar": "01", "februar": "02", ...}
```

Hvor langt kommer I? Der er ingen præmie for at få alle fire formater til at virke —
men skriv ned, hvilke I opgav og hvorfor.


In [ ]:
# Din kode her



### 3.3 Til diskussion

1. Hvor stor en andel af korpusset har til sidst en brugbar dato?
2. De dokumenter, der mangler dato, er ikke tilfældigt fordelt. Hvilke kilder
   forsvinder helt, hvis I laver en analyse over tid?
3. Er det bedre at lave analysen på halvdelen af materialet, eller at lade være?


---

## Opgave 4: Ordbogsmodellen

**Trin 1 — kør den fra slidesene.**


In [ ]:
ordbog = {
    "god": 2, "godt": 2, "bedre": 2, "bedst": 3, "stærk": 2,
    "sand": 2, "sandhed": 2, "frihed": 3, "tryghed": 2, "stolt": 2,
    "dårlig": -2, "værre": -2, "værst": -3, "katastrofe": -3,
    "løgn": -3, "svigt": -2, "trussel": -2, "farlig": -2,
    "krise": -2, "vold": -3, "frygt": -2, "svag": -2
}

def rens(t):
    t = str(t).lower()
    for tegn in [".", ",", "!", "?", ":", ";", '"', "(", ")"]:
        t = t.replace(tegn, " ")
    return t.split()

def score(tekst):
    return sum(ordbog.get(o, 0) for o in rens(tekst))

korpus["score"] = korpus["tekst"].apply(score)
print(korpus.groupby("kilde")["score"].median())


**Trin 2 — dækningen.**

Hvor mange dokumenter rammer ingen af de 22 ord? Regn det ud både som antal og som
andel.

Et dokument med score 0, fordi ordbogen ikke ramte, er ikke det samme som et neutralt
dokument. Men i tabellen ser de ens ud.


In [ ]:
# Din kode her



**Trin 3 — udvid listen.**

Tilføj 20-30 ord mere. Kig i materialet for at finde dem — brug `korpus["tekst"]`
og jeres ordoptælling fra lektion 2.

Hvor meget falder andelen af nul-scorer? Og hvad skete der med medianerne?


In [ ]:
# Din kode her



**Trin 4 — længden.**

Beregn antal ord per dokument. Lav derefter en normaliseret score: summen divideret
med antal ord, gange 1000.

Sammenlign medianerne per kilde før og efter normaliseringen. Hvilken kilde flytter
sig mest, og hvorfor netop den?


In [ ]:
# Din kode her



**Trin 5 — negation.**

Kør de tre sætninger. Skriv med ord, hvad modellen gør — ikke hvad den burde gøre.


In [ ]:
for s in ["Det er en god beslutning",
          "Det er ikke en god beslutning",
          "Ingen tryghed og ingen frihed"]:
    print(f"{s:34} {score(s):+d}")


### 4.6 Til diskussion

1. Kan negation løses med en større ordliste? Prøv at formulere en regel, der ville
   virke — og find derefter en sætning, hvor den fejler.
2. Hvis én kilde scorer mere negativt end en anden: hvad har I vist? Skriv de tre
   alternative forklaringer, I skal kunne udelukke først.
3. Hvad skulle der til, før I ville sætte et af tallene i en opgave?


---

## Opgave 5: spaCy

Herfra skal koden køres i UCloud eller lokalt. spaCy kan ikke installeres i browseren.

```bash
pip install spacy
python -m spacy download da_core_news_sm
```

**Trin 1 — kør pipelinen.**

Kør cellen, og skriv ned hvad I faktisk får. Slidesene viste jer koden, ikke svaret.


In [ ]:
import spacy

nlp = spacy.load("da_core_news_sm")
doc = nlp("Regeringen fremlagde et nyt udspil om indvandring i København.")

for tok in doc:
    print(tok.text, tok.lemma_, tok.pos_, tok.is_stop)


**Trin 2 — find en fejl.**

Modellen er trænet på dansk nyhedstekst. Jeres materiale er ikke nyhedstekst.

Kør pipelinen på tre-fire dokumenter fra korpusset, og find mindst ét sted, hvor
lemmatiseringen eller ordklassen er forkert.

Hvad slags ord går det galt på?


In [ ]:
# Din kode her



**Trin 3 — entiteter.**

`doc.ents` giver de navngivne entiteter: personer, steder, organisationer.

Kør den på 100 dokumenter fra én kilde, og tæl hvilke entiteter der optræder oftest.

*Hint:* `for ent in doc.ents: print(ent.text, ent.label_)`


In [ ]:
# Din kode her



**Trin 4 — tokenisering.**

Kør de to linjer på samme tekst og sammenlign resultaterne.

Hvad sker der med tankestreger, anførselstegn og udråbstegn i hver af dem?


In [ ]:
tekst = "Regeringens plan — den såkaldte 'løsning' — virker ikke!"

print(tekst.lower().split())
print([tok.text for tok in nlp(tekst)])


**Trin 5 — hvorfor det betyder noget for tallene.**

Beregn leksikalsk diversitet på 200 dokumenter — antal unikke ord divideret med antal
ord i alt — på to måder:

- med `.lower().split()`
- med spaCy-lemmaer, uden tegnsætning

Hvor stor bliver forskellen? Og hvilket af de to tal ville I skrive i en opgave?


In [ ]:
# Din kode her



---

## Opgave 6: Til portfolien

Vælg **ét** af de seks datasæt, eller jeres eget materiale.

1. Besvar Hunstons fire spørgsmål skriftligt. Kan I ikke svare på alle fire, er
   korpusset ikke færdigt.

2. Formulér ét forskningsspørgsmål, korpusset kan besvare — og ét, der ligner, men
   som det ikke kan.

3. Vælg én metode fra Macanovics fem familier, og skriv to sætninger om, hvorfor
   netop den passer til jeres spørgsmål.

4. Skriv de forbehandlingsvalg ned, I har truffet indtil nu. Alle sammen.


In [ ]:
# Din kode her



---

## Ekstraopgaver

**A. Ordforråd per kilde.** Tæl de 20 hyppigste ord i hver kilde, efter at stopord er
fjernet. Hvor meget overlapper listerne? Hvad adskiller kilderne?

**B. Længde og indhold.** Trykkefrihed skriver fire gange så lange indlæg som Dansk
Samling. Undersøg om det er én forfatter, én genre, eller noget tredje.

**C. Dubletter.** Er der dokumenter, der optræder to gange, eller tekster der er
næsten identiske? Hvordan ville I finde dem, og hvad betyder det for optællingerne?

**D. Over tid.** Brug de dokumenter, der har en dato. Tæl antal indlæg per måned per
kilde og tegn det med plotnine. Hvad kan figuren ikke vise?

**E. AFINN.** Installér `afinn`, kør `Afinn(language="da")` på korpusset, og
sammenlign med jeres egen ordbog. Hvor enige er de to modeller?


In [ ]:
# Din kode her

